# End-to-End LLM Training Tutorial (Offline)

This notebook demonstrates a complete offline workflow with this repository:
1. Train a tokenizer
2. Pretrain a tiny GPT model on toy data
3. Evaluate perplexity + probes
4. Run a tiny SFT pass
5. Optionally run a tiny DPO pass
6. Load the final checkpoint and generate text

> This is a toy educational run, not a frontier-scale recipe.

## 0) Setup

In [ ]:
from __future__ import annotations

import copy
import json
import random
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from tokenizers import Tokenizer as HFTokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer

# Resolve repo root robustly whether notebook is opened from repo root or notebooks/.
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'llmstack').exists():
            return p
    raise RuntimeError('Could not find repo root containing pyproject.toml and src/llmstack')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('Repo root:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
import tokenizers
print('Tokenizers:', tokenizers.__version__)

In [ ]:
from llmstack.data.datamodule import DataModule
from llmstack.eval.perplexity import evaluate_perplexity
from llmstack.eval.probes import run_probes
from llmstack.model.gpt import GPTModel
from llmstack.optim.adamw import build_adamw
from llmstack.optim.schedulers import build_scheduler
from llmstack.posttrain.dpo_trainer import dpo_loss
from llmstack.posttrain.sft_trainer import sft_step
from llmstack.tokenization.tokenizer import Tokenizer
from llmstack.train.engine import Trainer
from llmstack.utils.logging import RunLogger
from llmstack.utils.seed import seed_everything

In [ ]:
# Determinism + device/precision selection
SEED = 1234
seed_everything(SEED, deterministic=False)

if torch.cuda.is_available():
    DEVICE = 'cuda'
    PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
else:
    DEVICE = 'cpu'
    PRECISION = 'fp32'

print('DEVICE:', DEVICE, '| PRECISION:', PRECISION)

# Output layout
ARTIFACTS_DIR = REPO_ROOT / 'artifacts'
RUNS_DIR = REPO_ROOT / 'runs'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S_tutorial')
RUN_DIR = RUNS_DIR / RUN_NAME
CKPT_DIR = RUN_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_DIR:', RUN_DIR)

## 1) Train a Byte-level BPE tokenizer

In [ ]:
train_jsonl = REPO_ROOT / 'data' / 'toy' / 'train.jsonl'
assert train_jsonl.exists(), f'Missing {train_jsonl}'

# Build a plain-text corpus from JSONL text field
corpus_txt = ARTIFACTS_DIR / 'tokenizer_corpus.txt'
with train_jsonl.open('r', encoding='utf-8') as f_in, corpus_txt.open('w', encoding='utf-8') as f_out:
    for line in f_in:
        line = line.strip()
        if not line:
            continue
        f_out.write(json.loads(line)['text'] + '\n')

tok_out = ARTIFACTS_DIR / 'tokenizer_toy'
tok_out.mkdir(parents=True, exist_ok=True)

hf_tok = HFTokenizer(BPE(unk_token='<unk>'))
hf_tok.pre_tokenizer = ByteLevel()
trainer = BpeTrainer(
    vocab_size=1024,
    special_tokens=['<pad>', '<bos>', '<eos>', '<unk>'],
)
hf_tok.train([str(corpus_txt)], trainer=trainer)
hf_tok.save(str(tok_out / 'tokenizer.json'))

(tok_out / 'manifest.json').write_text(json.dumps({'vocab_size': hf_tok.get_vocab_size()}, indent=2), encoding='utf-8')

tokenizer = Tokenizer(tok_out)
print('Tokenizer vocab size:', tokenizer.vocab_size)
print('Special IDs:', tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id, tokenizer.unk_id)

## 2) Pretrain a small GPT model

In [ ]:
# Keep runtime modest; tweak these in one place.
PRETRAIN_STEPS = 80
SFT_STEPS = 30
DPO_STEPS = 20
RUN_DPO = True  # toggle off if you want the shortest runtime

# Tiny tutorial config using repository Trainer/DataModule APIs
cfg = SimpleNamespace(
    model=SimpleNamespace(
        vocab_size=tokenizer.vocab_size,
        n_layers=4,
        n_heads=4,
        d_model=256,
        d_ff=1024,
        max_seq_len=128,
        dropout=0.1,
        norm_type='layernorm',
        rope_enabled=False,
        tie_embeddings=True,
        gradient_checkpointing=False,
    ),
    data=SimpleNamespace(
        train_path='data/toy/train.jsonl',
        val_path='data/toy/val.jsonl',
        format='jsonl',
        text_field='text',
        seq_len=128,
        pack_sequences=True,
        num_workers=0,
        shuffle=True,
        streaming=False,
        add_bos=True,
        add_eos=True,
        pad_to_seq_len=True,
    ),
    train=SimpleNamespace(
        seed=SEED,
        device=DEVICE,
        precision=PRECISION,
        compile=False,
        grad_clip=1.0,
        grad_accum_steps=1,
        micro_batch_size=4 if DEVICE == 'cpu' else 8,
        max_steps=PRETRAIN_STEPS,
        eval_interval=max(10, PRETRAIN_STEPS // 4),
        log_interval=max(5, PRETRAIN_STEPS // 10),
        save_interval=max(10, PRETRAIN_STEPS // 2),
        out_dir=str(RUNS_DIR),
        run_name='tutorial_pretrain',
        resume_path=None,
        max_eval_batches=20,
        deterministic=False,
    ),
    optim=SimpleNamespace(
        name='adamw',
        lr=3e-4,
        betas=(0.9, 0.95),
        weight_decay=0.1,
        eps=1e-8,
    ),
    sched=SimpleNamespace(name='warmup_cosine', warmup_steps=10, min_lr=0.1),
    log=SimpleNamespace(use_wandb=False, wandb_project='llmstack', jsonl=True, tensorboard=True),
    run_dir=str(CKPT_DIR),
)

seed_everything(cfg.train.seed, deterministic=cfg.train.deterministic)
dm = DataModule(cfg, tokenizer, world_size=1, rank=0)
model = GPTModel(cfg.model).to(DEVICE)
optimizer = build_adamw(model, cfg.optim)
scheduler = build_scheduler(optimizer, cfg.sched, cfg.train.max_steps)
logger = RunLogger(str(RUN_DIR), jsonl=cfg.log.jsonl, tensorboard=cfg.log.tensorboard)
trainer = Trainer(cfg, model, optimizer, scheduler, logger, dm.train_dataloader(), dm.val_dataloader(), device=DEVICE)

trainer.fit()
trainer.save_checkpoint('last.pt')
logger.close()

PRETRAIN_CKPT = CKPT_DIR / 'last.pt'
assert PRETRAIN_CKPT.exists()
print('Saved pretrain checkpoint:', PRETRAIN_CKPT)

In [ ]:
# Export model-only weights (equivalent to scripts/export_checkpoint.py functionality)
export_path = CKPT_DIR / 'pretrain_model_state.pt'
state = torch.load(PRETRAIN_CKPT, map_location='cpu')
torch.save(state['model'], export_path)
print('Exported model state_dict to:', export_path)

## 3) Evaluate perplexity + probes

In [ ]:
@torch.no_grad()
def run_eval(ckpt_path: Path, label: str) -> dict:
    eval_model = GPTModel(cfg.model).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    eval_model.load_state_dict(ckpt['model'])
    eval_model.eval()

    val_loader = dm.val_dataloader()
    ppl = evaluate_perplexity(eval_model, val_loader, DEVICE, max_batches=cfg.train.max_eval_batches)
    probes = run_probes(eval_model, tokenizer, DEVICE)

    report = {'label': label, 'checkpoint': str(ckpt_path), 'perplexity': ppl, 'probes': probes}
    out = RUN_DIR / f'eval_{label}.json'
    out.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report

pretrain_report = run_eval(PRETRAIN_CKPT, 'pretrain')

## 4) Tiny SFT example

In [ ]:
# Create tiny SFT dataset offline
sft_path = REPO_ROOT / 'data' / 'toy' / 'sft.jsonl'
sft_examples = [
    {'prompt': 'Q: What is 2 + 2?', 'response': '4'},
    {'prompt': 'Q: Say hello politely.', 'response': 'Hello! Nice to meet you.'},
    {'prompt': 'Q: Complete: The sky is', 'response': ' blue.'},
]
with sft_path.open('w', encoding='utf-8') as f:
    for ex in sft_examples:
        f.write(json.dumps(ex) + '\n')
print('Wrote SFT data:', sft_path)

# Load model from pretrain checkpoint
sft_model = GPTModel(cfg.model).to(DEVICE)
sft_model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=DEVICE)['model'])
sft_model.train()

sft_optimizer = torch.optim.AdamW(sft_model.parameters(), lr=2e-4)
separator = '\n### Response:\n'
rng = random.Random(SEED)

for step in range(SFT_STEPS):
    ex = sft_examples[rng.randrange(len(sft_examples))]
    loss = sft_step(sft_model, tokenizer, [ex], separator=separator, device=DEVICE)
    sft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
    sft_optimizer.step()
    if (step + 1) % max(1, SFT_STEPS // 5) == 0:
        print(f'SFT step {step+1}/{SFT_STEPS} loss={loss.item():.4f}')

SFT_CKPT = CKPT_DIR / 'sft_last.pt'
torch.save({'model': sft_model.state_dict(), 'step': SFT_STEPS}, SFT_CKPT)
print('Saved SFT checkpoint:', SFT_CKPT)

In [ ]:
sft_report = run_eval(SFT_CKPT, 'sft')

## 5) Tiny DPO example (optional)

In [ ]:
dpo_path = REPO_ROOT / 'data' / 'toy' / 'prefs.jsonl'
dpo_examples = [
    {'prompt': 'Complete: 2+2=', 'chosen': ' 4', 'rejected': ' 5'},
    {'prompt': 'Answer yes/no: Is water wet?', 'chosen': ' yes', 'rejected': ' no'},
    {'prompt': 'Complete: opposite of hot is', 'chosen': ' cold', 'rejected': ' spicy'},
]
with dpo_path.open('w', encoding='utf-8') as f:
    for ex in dpo_examples:
        f.write(json.dumps(ex) + '\n')
print('Wrote DPO data:', dpo_path)

FINAL_CKPT = SFT_CKPT

if RUN_DPO:
    policy = GPTModel(cfg.model).to(DEVICE)
    policy.load_state_dict(torch.load(SFT_CKPT, map_location=DEVICE)['model'])
    policy.train()

    # Reference model (copy-at-start behavior in notebook)
    ref = copy.deepcopy(policy).to(DEVICE)
    ref.eval()
    for p in ref.parameters():
        p.requires_grad = False

    dpo_optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-5)
    beta = 0.1
    for step in range(DPO_STEPS):
        ex = dpo_examples[step % len(dpo_examples)]
        loss = dpo_loss(policy, ref, tokenizer, ex, beta=beta, device=DEVICE)
        dpo_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        dpo_optimizer.step()
        if (step + 1) % max(1, DPO_STEPS // 5) == 0:
            print(f'DPO step {step+1}/{DPO_STEPS} loss={loss.item():.4f}')

    DPO_CKPT = CKPT_DIR / 'dpo_last.pt'
    torch.save({'model': policy.state_dict(), 'step': DPO_STEPS}, DPO_CKPT)
    FINAL_CKPT = DPO_CKPT
    print('Saved DPO checkpoint:', DPO_CKPT)
    dpo_report = run_eval(DPO_CKPT, 'dpo')
else:
    print('Skipping DPO because RUN_DPO=False')

## 6) Minimal inference / generation demo

In [ ]:
@torch.no_grad()
def greedy_generate(model: GPTModel, tok: Tokenizer, prompt: str, max_new_tokens: int = 32) -> str:
    model.eval()
    ids = tok.encode(prompt)
    if len(ids) == 0:
        ids = [tok.bos_id]
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)

    for _ in range(max_new_tokens):
        x_cond = x[:, -cfg.model.max_seq_len :]
        logits, _ = model(x_cond)
        next_id = int(torch.argmax(logits[:, -1, :], dim=-1).item())
        x = torch.cat([x, torch.tensor([[next_id]], device=DEVICE)], dim=1)
        if next_id == tok.eos_id:
            break

    return tok.decode(x[0].tolist())

final_model = GPTModel(cfg.model).to(DEVICE)
final_model.load_state_dict(torch.load(FINAL_CKPT, map_location=DEVICE)['model'])

prompt = 'Q: What is 2 + 2?\n### Response:\n'
print('Prompt:', prompt)
print('Generated:', greedy_generate(final_model, tokenizer, prompt, max_new_tokens=24))
print('\nNote: This is a tiny toy model trained very briefly; output quality is limited.')

## Outputs

- Tokenizer artifacts: `artifacts/tokenizer_toy/`
- Run logs + eval reports: `runs/<timestamp>_tutorial/`
- Checkpoints: `runs/<timestamp>_tutorial/checkpoints/`

You can re-run with different step counts by editing `PRETRAIN_STEPS`, `SFT_STEPS`, `DPO_STEPS`, and `RUN_DPO`.